In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for d, _, fs in os.walk('/kaggle/input'):
    for f in fs: print(os.path.join(d, f))

In [ ]:
!pip install skorch imbalanced-learn -q

import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
from skorch import NeuralNetClassifier

os.makedirs("/kaggle/working/images", exist_ok=True)
torch.manual_seed(42); np.random.seed(42)

In [ ]:
path = "/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv"
df = pd.read_csv(path)
print(df.shape)
df.info()
print(df.isnull().sum())
df.head()

In [ ]:
print(df["Failure Type"].value_counts())
print()
print(pd.crosstab(df["Target"], df["Failure Type"]))

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(y="Failure Type", data=df, order=df["Failure Type"].value_counts().index)
plt.title("Distribution des classes (Failure Type)")
plt.savefig("/kaggle/working/images/p2_classes.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
num = ["Air temperature [K]", "Process temperature [K]",
       "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, c in zip(ax.ravel(), num):
    sns.boxplot(x="Failure Type", y=c, data=df, ax=a)
    a.tick_params(axis="x", rotation=60); a.set_title(c)
ax.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/images/p2_boxplots.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[num].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Corrélations")
plt.savefig("/kaggle/working/images/p2_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
d = df.copy()

# 1. supprimer les identifiants
d = d.drop(columns=["UDI", "Product ID"])

# 2. supprimer les lignes incohérentes
bad = ((d["Target"] == 1) & (d["Failure Type"] == "No Failure")) | \
      ((d["Target"] == 0) & (d["Failure Type"] != "No Failure"))
print("Lignes incohérentes supprimées :", bad.sum())
d = d[~bad].reset_index(drop=True)

# 3. encodage
d["Type"] = d["Type"].map({"L": 0, "M": 1, "H": 2})
le = LabelEncoder()
y = le.fit_transform(d["Failure Type"])
X = d.drop(columns=["Failure Type", "Target"])
print(dict(zip(le.classes_, range(len(le.classes_)))))
print(X.shape, np.bincount(y))

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X.values, y, test_size=0.2, stratify=y, random_state=42)

sc = StandardScaler()
X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

print("Avant SMOTE :", np.bincount(y_tr))
k = max(1, min(3, np.bincount(y_tr).min() - 1))
sm = SMOTE(k_neighbors=k, random_state=42)
X_tr_bal, y_tr_bal = sm.fit_resample(X_tr, y_tr)
print("Après SMOTE :", np.bincount(y_tr_bal))

X_tr_bal = X_tr_bal.astype("float32"); X_te = X_te.astype("float32")
y_tr_bal = y_tr_bal.astype("int64");   y_te = y_te.astype("int64")

In [ ]:
n_classes = len(le.classes_)

class MLPClf(nn.Module):
    def __init__(self, in_features=6, hidden=64, n_classes=5,
                 dropout=0.0, batchnorm=False):
        super().__init__()
        layers = [nn.Linear(in_features, hidden)]
        if batchnorm: layers.append(nn.BatchNorm1d(hidden))
        layers += [nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, hidden // 2)]
        if batchnorm: layers.append(nn.BatchNorm1d(hidden // 2))
        layers += [nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden // 2, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

m = MLPClf(in_features=X_tr_bal.shape[1], n_classes=n_classes)
print(m)

In [ ]:
net = NeuralNetClassifier(
    MLPClf, module__in_features=X_tr_bal.shape[1], module__n_classes=n_classes,
    criterion=nn.CrossEntropyLoss, optimizer=torch.optim.Adam,
    max_epochs=20, batch_size=128, verbose=0, train_split=None)

params = {
    "lr": [1e-2, 1e-3],
    "optimizer": [torch.optim.Adam, torch.optim.SGD],
    "max_epochs": [20, 40],
    "module__hidden": [32, 64],
}
gs = GridSearchCV(net, params, cv=3, scoring="f1_macro", n_jobs=1)
gs.fit(X_tr_bal, y_tr_bal)
print("Meilleurs paramètres :", gs.best_params_)
print("Meilleur F1 macro (CV) :", gs.best_score_)

In [ ]:
res = pd.DataFrame(gs.cv_results_)
res["optimizer"] = res["param_optimizer"].apply(lambda o: o.__name__)
tab = res[["param_lr", "optimizer", "param_max_epochs", "param_module__hidden",
           "mean_test_score", "rank_test_score"]]
print(tab.sort_values("rank_test_score").head(8).round(4).to_string(index=False))

In [ ]:
def train_clf(model, lr, optimizer_cls, epochs, weight_decay=0.0,
              patience=None, batch_size=128):
    Xtr, ytr = torch.tensor(X_tr_bal), torch.tensor(y_tr_bal)
    Xte, yte = torch.tensor(X_te), torch.tensor(y_te)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    opt = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()
    h = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}
    best, wait = float("inf"), 0
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            ptr, pte = model(Xtr), model(Xte)
            h["train_loss"].append(crit(ptr, ytr).item())
            h["test_loss"].append(crit(pte, yte).item())
            h["train_acc"].append((ptr.argmax(1) == ytr).float().mean().item())
            h["test_acc"].append((pte.argmax(1) == yte).float().mean().item())
        if patience:
            if h["test_loss"][-1] < best - 1e-6:
                best, wait = h["test_loss"][-1], 0
            else:
                wait += 1
                if wait >= patience:
                    break
    return h

def plot_clf(h, title="", save_as=None):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["test_loss"], label="test")
    ax[0].set_title(f"Loss / Epochs {title}"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["train_acc"], label="train"); ax[1].plot(h["test_acc"], label="test")
    ax[1].set_title(f"Accuracy / Epochs {title}"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save_as:
        plt.savefig(f"/kaggle/working/images/{save_as}", dpi=120, bbox_inches="tight")
    plt.show()

def metrics(model, X, y):
    model.eval()
    with torch.no_grad():
        p = model(torch.tensor(X)).argmax(1).numpy()
    return {"Accuracy": accuracy_score(y, p),
            "Précision (macro)": precision_score(y, p, average="macro", zero_division=0),
            "Sensibilité/Recall (macro)": recall_score(y, p, average="macro", zero_division=0),
            "F1 (macro)": f1_score(y, p, average="macro", zero_division=0)}, p

In [ ]:
bp = gs.best_params_
torch.manual_seed(42)
base = MLPClf(in_features=X_tr_bal.shape[1], hidden=bp["module__hidden"], n_classes=n_classes)
h_base = train_clf(base, bp["lr"], bp["optimizer"], bp["max_epochs"])
plot_clf(h_base, "(baseline)", save_as="p2_courbes_baseline.png")

In [ ]:
for name in ["X_tr_bal", "y_tr_bal", "X_te", "y_te", "MLPClf", "gs", "train_clf", "plot_clf", "metrics"]:
    print(name, "OK" if name in globals() else "MANQUANT")

In [ ]:
print("Epochs :", len(h_base["train_loss"]))
print("Loss train/test epoch 1 :", h_base["train_loss"][0], h_base["test_loss"][0])
print("Loss train/test finale  :", h_base["train_loss"][-1], h_base["test_loss"][-1])
print("Acc train/test epoch 1  :", h_base["train_acc"][0], h_base["test_acc"][0])
print("Acc train/test finale   :", h_base["train_acc"][-1], h_base["test_acc"][-1])

In [ ]:
m_tr, _ = metrics(base, X_tr_bal, y_tr_bal)
m_te, p_te = metrics(base, X_te, y_te)
print(pd.DataFrame({"Train": m_tr, "Test": m_te}).round(4))
print()
print(classification_report(y_te, p_te, target_names=le.classes_, zero_division=0))

In [ ]:
cm = confusion_matrix(y_te, p_te)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Prédit"); plt.ylabel("Réel"); plt.title("Matrice de confusion (test)")
plt.savefig("/kaggle/working/images/p2_confusion.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
configs = {
    "Baseline":       dict(model=dict(), wd=0.0, pat=None),
    "Dropout 0.2":    dict(model=dict(dropout=0.2), wd=0.0, pat=None),
    "L2 (wd=1e-4)":   dict(model=dict(), wd=1e-4, pat=None),
    "BatchNorm":      dict(model=dict(batchnorm=True), wd=0.0, pat=None),
    "Early stopping": dict(model=dict(), wd=0.0, pat=5),
    "Combiné":        dict(model=dict(dropout=0.2), wd=1e-4, pat=5),
}
results, hists = [], {}
for name, c in configs.items():
    torch.manual_seed(42)
    m = MLPClf(in_features=X_tr_bal.shape[1], hidden=bp["module__hidden"],
               n_classes=n_classes, **c["model"])
    h = train_clf(m, bp["lr"], bp["optimizer"], bp["max_epochs"],
                  weight_decay=c["wd"], patience=c["pat"])
    hists[name] = h
    mt, _ = metrics(m, X_tr_bal, y_tr_bal)
    me, _ = metrics(m, X_te, y_te)
    results.append({"Modèle": name, "Epochs": len(h["train_loss"]),
                    "Train Acc": mt["Accuracy"], "Test Acc": me["Accuracy"],
                    "Test Précision macro": me["Précision (macro)"],
                    "Test Recall macro": me["Sensibilité/Recall (macro)"],
                    "Test F1 macro": me["F1 (macro)"]})
comparison = pd.DataFrame(results).round(4)
print(comparison.to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 5))
for name, h in hists.items():
    plt.plot(h["test_loss"], label=name)
plt.xlabel("epoch"); plt.ylabel("test loss")
plt.title("Comparaison des techniques de régularisation")
plt.legend()
plt.savefig("/kaggle/working/images/p2_comparaison_reg.png", dpi=120, bbox_inches="tight")
plt.show()